In [1]:
import os

from IPython.display import FileLink

if os.getcwd() == '/notebooks':
    os.chdir("./motion-synthesis")
    print('inside dir: ', os.listdir())
print(os.listdir())
print("Click here to download the dit_d0.tar: ", display(FileLink("/notebooks/motion-synthesis/dit_d0.tar")))


os.environ["TOKENIZERS_PARALLELISM"] = "false"

['README_v1.md', 'options', 'main.ipynb', 'conda_requirements.txt', '.DS_Store', 'requirements.txt', 'prepare', 'eval_models', 'lambda_requirements.txt', 'data_utils', 'checkpoints', 'utils', 'README.md', 'dataset_split_builder.py', 'networks', 'common', 'glove', '.gitignore', 'log', 'environment_cpu.yaml', 'environment.yaml', 'exp_results', '.git', '.vscode', 'main.py', 'demo.py', 'data', 'assets', 'outputs']


/notebooks/motion-synthesis/dit_d0.tar

Click here to download the dit_d0.tar:  None


In [2]:
import torch
from torch.backends import cuda
from options.train_options  import TrainOptions
from os.path import join as pjoin
import os
from utils.paramUtils import t2m_kinematic_chain
import numpy as np
from utils.word_vectorizer import WordVectorizer
from torch.utils.data import DataLoader
from data_utils.dataset import MotionDatasetV2
from data_utils.dataset import PartMotionDatasetV2
from networks.nn import MotionVQVAE, DiT
from networks.trainers import MotionVQVAETrainer, MotionDiTTrainer
from torch.utils.data import Subset
from networks.nn_validator import VQVAEValidator, DiffusionValidator
import json

/Users/beethoven/miniforge3/envs/vper-motion-editing/lib/python3.11/site-packages/torchvision/io/image.py:14: UserWarning: Failed to load image Python extension: 'dlopen(/Users/beethoven/miniforge3/envs/vper-motion-editing/lib/python3.11/site-packages/torchvision/image.so, 0x0006): Library not loaded: @rpath/libjpeg.9.dylib
  Referenced from: <EB3FF92A-5EB1-3EE8-AF8B-5923C1265422> /Users/beethoven/miniforge3/envs/vper-motion-editing/lib/python3.11/site-packages/torchvision/image.so
  Reason: tried: '/Users/beethoven/miniforge3/envs/vper-motion-editing/lib/python3.11/site-packages/torchvision/../../../libjpeg.9.dylib' (no such file), '/Users/beethoven/miniforge3/envs/vper-motion-editing/lib/python3.11/site-packages/torchvision/../../../libjpeg.9.dylib' (no such file), '/Users/beethoven/miniforge3/envs/vper-motion-editing/lib/python3.11/lib-dynload/../../libjpeg.9.dylib' (no such file), '/Users/beethoven/miniforge3/envs/vper-motion-editing/bin/../lib/libjpeg.9.dylib' (no such file)'If yo

In [3]:
parser = TrainOptions()
options = parser.parse(args = ['--max_epoch', '10', '--lr', '1e-4', '--save_latest', '10', '--eval_every_e', '1', '--save_every_e', '20', '--log_every', '1'])
options.gpu_id = torch.cuda.current_device() if torch.cuda.is_available() else -1
options.device = torch.device("cpu" if options.gpu_id==-1 else "cuda:" + str(options.gpu_id))
torch.autograd.set_detect_anomaly(True)

# disabling flash backend till the architecture parameters allow stable use of flash backend for attention
# without creating nans
cuda.enable_flash_sdp(False)
cuda.enable_mem_efficient_sdp(False)
cuda.enable_math_sdp(True)

if options.gpu_id != -1:
    # self.opt.gpu_id = int(self.opt.gpu_id)
    torch.cuda.set_device(options.gpu_id)

print('\nDevice used: ', options.device)
options.save_root = pjoin(options.checkpoints_dir, 'HumanML3D', options.name)
options.model_dir = pjoin(options.checkpoints_dir, 'model')
options.meta_dir = pjoin(options.save_root, 'meta')
options.eval_dir = pjoin(options.save_root, 'animation')
options.log_dir = pjoin('./log', options.dataset_name, options.name)
options.experiment_dir = './exp_results/stable-diffusion-setup/finetune_dit_cross_attn'
options.output_dir = options.experiment_dir
options.is_train = False
options.is_continue = False
options.dataset_mode = "nano"
options.batch_size = 64
options.model_filename = 'dit_stable_crossattn_nano.tar'

os.makedirs(options.model_dir, exist_ok=True)
os.makedirs(options.meta_dir, exist_ok=True)
os.makedirs(options.eval_dir, exist_ok=True)
os.makedirs(options.log_dir, exist_ok=True)

options.data_root = './data/HumanML3D'
options.motion_dir = pjoin(options.data_root, 'new_joint_vecs')
options.text_dir = pjoin(options.data_root, 'texts')
options.joints_num = 22
options.max_motion_length = 120
dim_pose = 263
radius = 4
fps = 20
kinematic_chain = t2m_kinematic_chain

print('Set parameters')
print('Learning rate: ', options.lr)
print('Max epochs: ', options.max_epoch)
print('Save latest frequency: ', options.save_latest)
print('Save every epoch frequency: ', options.save_every_e)
print('Log every iterations frequency: ', options.log_every)


Device used:  cpu
Set parameters
Learning rate:  0.0001
Max epochs:  10
Save latest frequency:  10
Save every epoch frequency:  20
Log every iterations frequency:  1


In [4]:
mean = np.load(pjoin(options.data_root, 'Mean.npy'))
std = np.load(pjoin(options.data_root, 'Std.npy'))

w_vectorizer = WordVectorizer('./glove', 'our_vab')
train_split_fn = 'train.txt'
val_split_fn = 'val.txt'
test_split_fn = 'test.txt'
simple_test_split_fn = 'simple_test.txt'

if options.dataset_mode in ["debug", "nano", "micro"]:
    train_split_fn = f'train_{options.dataset_mode}.txt'
    val_split_fn = f'val_{options.dataset_mode}.txt'

train_split_file = pjoin(options.data_root, train_split_fn)
val_split_file = pjoin(options.data_root, val_split_fn)
test_split_file = pjoin(options.data_root, test_split_fn)
simple_test_split_file = pjoin(options.data_root, simple_test_split_fn)

train_dataset = PartMotionDatasetV2(options, mean, std, train_split_file)
val_dataset = PartMotionDatasetV2(options, mean, std, val_split_file)
test_dataset = PartMotionDatasetV2(options, mean, std, test_split_file)

print('\nTotal number of snippets in train: ', len(train_dataset))
print('Total number of snippets in val: ', len(val_dataset))
print('Total number of snippets in test: ', len(test_dataset))

sample_motion = train_dataset[4]
print('Sample data shape: ', sample_motion['motion_parts'].shape, sample_motion['text'])
Dp_max = sample_motion['motion_parts'].shape[-1]

id list 32


100%|██████████| 32/32 [00:00<00:00, 1044.38it/s]


Motion shape (B, T, D): (96, 120, 263)
Total number of motions 96
Total number of small motions: 17
id list 16


100%|██████████| 16/16 [00:00<00:00, 1222.41it/s]


Motion shape (B, T, D): (48, 120, 263)
Total number of motions 48
Total number of small motions: 7
id list 4384


100%|██████████| 4384/4384 [00:02<00:00, 1600.48it/s]

Motion shape (B, T, D): (13114, 120, 263)
Total number of motions 13114
Total number of small motions: 1710

Total number of snippets in train:  96
Total number of snippets in val:  48
Total number of snippets in test:  13114
Sample data shape:  (120, 6, 60) the person is jumping up and down.


In [5]:
train_loader = DataLoader(train_dataset, batch_size=options.batch_size, drop_last=not(options.dataset_mode in ['micro', 'nano']), num_workers=1,
                              shuffle=True, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=options.batch_size, drop_last=not(options.dataset_mode in ['micro', 'nano']), num_workers=1,
                        shuffle=True, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=options.batch_size, drop_last=not(options.dataset_mode in ['micro', 'nano']), num_workers=1,
                         shuffle=True, pin_memory=True)

if options.stage == "autoencoder":
    vqvae = MotionVQVAE(
        input_dim=Dp_max,
        enc_hidden_dim=1024,
        dec_hidden_dim=1024,
        latent_dim=256,
        num_embeddings=512,
        beta=0.1
    )

    if options.is_train:
        trainer = MotionVQVAETrainer(options, vqvae = vqvae)
        trainer.train(
            train_dataloader=train_loader,
            val_dataloader=val_loader)
else:
    dit = DiT(
        input_size = 512,
        hidden_size = 1152,
        text_dim = 768,
        max_seq_len=options.max_motion_length // 4
    )

    if options.is_train:
        trainer = MotionDiTTrainer(
            args = options,
            dit = dit,
            autoencoder_type="pretrained_vae"
        )
        trainer.train(
            train_dataloader=train_loader,
            val_dataloader=val_loader
        )


In [7]:
if options.stage == "autoencoder":
    test_model_filepath = pjoin(options.model_dir, options.model_filename)

    if not(options.is_train) and os.path.exists(test_model_filepath):
        vqvae_model_dict = torch.load(test_model_filepath, map_location = options.device)

        vqvae.load_state_dict(vqvae_model_dict['vqvae'])

        vqvae_validator = VQVAEValidator(
            opt = options,
            vqvae=vqvae,
            train_dataloader=train_loader,
            val_dataloader = val_loader
        )
        vqvae_validator.validate()
    else:
        print("Invalid mode or model file doesn't exist!")
else:
    test_model_filepath = pjoin(options.model_dir, options.model_filename)
    if not(options.is_train) and os.path.exists(test_model_filepath):
        dit_model_dict = torch.load(test_model_filepath, map_location = options.device)

        dit.load_state_dict(dit_model_dict['dit'])

        dit_validator = DiffusionValidator(
            opt = options,
            dit=dit,
            val_dataloader = val_loader,
            test_dataloader=test_loader,
            test_type = "test_simple"
        )
        dit_validator.validate()
    else:
        print("Invalid mode or model file doesn't exist!")

/var/folders/47/n3y2_kpd3qz8ch8n8rgxpyjm0000gn/T/ipykernel_35071/3193172733.py:21: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  dit_model_dict = torch.load(test_model_filep

Reconstructed motion shape: torch.Size([5, 120, 263]) torch.Size([29, 10, 768])
joints recon torch.Size([120, 22, 3]) ./exp_results/stable-diffusion-setup/finetune_dit_cross_attn/videos
Saved video file at:  ./exp_results/stable-diffusion-setup/finetune_dit_cross_attn/videos/prompt_0.gif
joints recon torch.Size([120, 22, 3]) ./exp_results/stable-diffusion-setup/finetune_dit_cross_attn/videos
Saved video file at:  ./exp_results/stable-diffusion-setup/finetune_dit_cross_attn/videos/prompt_1.gif
joints recon torch.Size([120, 22, 3]) ./exp_results/stable-diffusion-setup/finetune_dit_cross_attn/videos
Saved video file at:  ./exp_results/stable-diffusion-setup/finetune_dit_cross_attn/videos/prompt_2.gif
joints recon torch.Size([120, 22, 3]) ./exp_results/stable-diffusion-setup/finetune_dit_cross_attn/videos
Saved video file at:  ./exp_results/stable-diffusion-setup/finetune_dit_cross_attn/videos/prompt_3.gif
joints recon torch.Size([120, 22, 3]) ./exp_results/stable-diffusion-setup/finetune_